In [ ]:
from utils.postgres_util import get_engine_from_env, read_sql_dataframe
from utils.row_rebuilder import rebuild_consumed_messages_to_observations


In [ ]:
SCHEMA = str("capstone")

DATASET_ID = str("pump_synthetic_v1")
RUN_ID = str("premelt_run_001")

REBUILD_CONSUMED_MESSAGES_SOURCE_TABLE_NAME = str("synthetic_sensor_messages_consumed_stage")
REBUILT_CONSUMED_MESSAGES_DESTINATION_TABLE_NAME = str("synthetic_sensor_observations_rebuilt_stage")


REBUILD_STATUS = str("pending")
NUMBER_OF_SENSORS = int(52)

COMPLETE_ONLY_FLAG = True
MARK_SOURCE_REBUILT_FLAG = True

OBSERVATION_WINDOW_SIZE = int(2500)

In [ ]:

engine = get_engine_from_env()


In [ ]:


rebuild_result = rebuild_consumed_messages_to_observations(
    engine=engine,
    schema=SCHEMA,
    source_table=REBUILD_CONSUMED_MESSAGES_SOURCE_TABLE_NAME,
    target_table=REBUILT_CONSUMED_MESSAGES_DESTINATION_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    rebuild_status=REBUILD_STATUS,
    n_sensors=NUMBER_OF_SENSORS,
    complete_only=COMPLETE_ONLY_FLAG,
    mark_source_rebuilt=MARK_SOURCE_REBUILT_FLAG,
    observation_window_size=OBSERVATION_WINDOW_SIZE,
)


In [ ]:

rebuild_result

----

In [ ]:
validation_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        COUNT(*) AS rebuilt_row_count,
        COUNT(*) FILTER (WHERE rebuild_is_complete = TRUE) AS complete_row_count,
        MIN(observation_index) AS min_observation_index,
        MAX(observation_index) AS max_observation_index,
        MIN(observation_timestamp) AS min_observation_timestamp,
        MAX(observation_timestamp) AS max_observation_timestamp
    FROM capstone.synthetic_sensor_observations_rebuilt_stage
    """
)


In [ ]:

display(validation_dataframe)

---

In [ ]:
sample_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        dataset_id,
        run_id,
        asset_id,
        observation_index,
        observation_timestamp,
        stream_state,
        phase,
        meta_episode_id,
        meta_primary_fault_type,
        sensor_00,
        sensor_01,
        sensor_02,
        sensor_03,
        sensor_04,
        rebuild_sensor_count,
        rebuild_is_complete
    FROM capstone.synthetic_sensor_observations_rebuilt_stage
    ORDER BY observation_index
    LIMIT 10
    """
)


In [ ]:

display(sample_dataframe)

----

In [ ]:
incomplete_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        dataset_id,
        run_id,
        asset_id,
        observation_index,
        rebuild_sensor_count,
        rebuild_is_complete,
        rebuild_notes
    FROM capstone.synthetic_sensor_observations_rebuilt_stage
    WHERE rebuild_is_complete = FALSE
    ORDER BY observation_index
    LIMIT 25
    """
)

display(incomplete_dataframe)

----